
# Execício Análise dados UEFA EURO

In [ ]:
# Fonte dos dados (Hugging Face): https://huggingface.co/datasets/debs-b/uefa-euro

In [1]:
from huggingface_hub import snapshot_download
import os
import pandas as pd
import glob
import matplotlib.pyplot as plt

In [ ]:
# Definir o ID do repositório no Hugging Face
repo_id = "debs-b/uefa-euro"

# O snapshot_download baixa e faz o cache dos dados automaticamente!
caminho_dataset = snapshot_download(repo_id=repo_id, repo_type="dataset")
print(f"Dataset pronto e cacheado em: {caminho_dataset}")

In [ ]:
# Célula substituída pela integração com o Hugging Face

In [ ]:
files = os.listdir(caminho_dataset)
files


In [ ]:
# Definir o caminho para os CSVs dentro da pasta euro
euro_path = os.path.join(caminho_dataset, "matches", "matches", "euro")

# Listar os arquivos CSV na pasta especificada
try:
    euro_files = [f for f in os.listdir(euro_path) if f.endswith('.csv')]
    euro_files
except FileNotFoundError as e:
    str(e)


euro_files

In [ ]:
# Carregando todos os arquivos em um único dataset para análise de dados


# Usar glob para localizar todos os arquivos CSV na pasta especificada
csv_files = glob.glob(os.path.join(caminho_dataset, "matches", "matches", "euro", "*.csv"))

# Carregar todos os arquivos CSV em um único DataFrame
dataframes = []
for file in csv_files:
    try:
        df = pd.read_csv(file)
        dataframes.append(df)
    except Exception as e:
        print(f"Erro ao carregar {file}: {e}")

# Concatenar todos os DataFrames em um único DataFrame
if dataframes:
    euro_data = pd.concat(dataframes, ignore_index=True)
    print("Dados carregados com sucesso!")
    euro_data.head()
else:
    print("Nenhum arquivo foi carregado.")


In [ ]:
euro_data.head(5)

In [ ]:
# Vamos ajustar a configuração do pandas?

pd.set_option('display.max_rows', None)  # Exibir todas as linhas
pd.set_option('display.max_columns', None)  # Exibir todas as colunas
pd.set_option('display.width', None)  # Ajustar a largura para não cortar dados
pd.set_option('display.colheader_justify', 'center')  # Justificar os cabeçalhos

# colocar esse código logo depois de dar 'import pandas as pd' em outros projetos! Ajuda a não esquecer

In [ ]:
euro_data.head(5)

In [ ]:
len(euro_data)

In [ ]:
len(euro_data.columns)

In [ ]:
euro_data.columns

# Análise Valores Nulos

In [ ]:
valores_nulos = euro_data.isnull().sum()
valores_nulos

## O que podemos observar baseado nas colunas que estão nulas?

In [ ]:
valores_nulos_totais = valores_nulos[valores_nulos > 0]
valores_nulos_totais

### Penaltis faz sentido;
### Winner tbm, por causa dos empates;
### alem da fase de grupos, nao tem mais grupo;
### condicoes climaticas --> o pessoal ficou com preguica de anotar, normal;
### gols tbm faz sentido, por causa das partidas de 0x0;
...

In [ ]:
registros_duplicados = euro_data.duplicated().sum()
registros_duplicados

# Análise Estatística 

In [ ]:
# Listar as colunas por tipo de dado

# Colunas categóricas
colunas_categoricas = euro_data.select_dtypes(include=['object']).columns.tolist()
print("Colunas categóricas:")
print(colunas_categoricas)

# Colunas numéricas
colunas_numericas = euro_data.select_dtypes(include=['number']).columns.tolist()
print("\nColunas numéricas:")
print(colunas_numericas)

# Colunas de data/hora
# Deixando aqui como exemplo para futuras análises
colunas_datahora = euro_data.select_dtypes(include=['datetime']).columns.tolist()
print("\nColunas de data/hora:")
print(colunas_datahora)



In [ ]:
# País mais vencedor

In [ ]:
vencedores = euro_data['winner'].value_counts()
print("Times mais vencedores da EURO (número de vitórias):")
print(vencedores)
print(f"\nTime mais vencedor: {vencedores.idxmax()} com {vencedores.max()} vitórias.")

### fui checar online, e Espanha realmente tem mais titulos, com 4. 
### Seguida por Alemanha, 3;  Italia 2; Franca 2....


In [ ]:
# Distribuição de gols por partida

In [ ]:
euro_data['total_goals'] = euro_data['home_score'] + euro_data['away_score']

# Calcular a média de gols
media_gols = euro_data['total_goals'].mean()

# Plotar a distribuição com a média destacada
plt.figure(figsize=(10, 6))
max_goals = int(euro_data['total_goals'].max()) + 1  # Garantir que o limite seja um inteiro
plt.hist(euro_data['total_goals'], bins=range(0, max_goals + 1), edgecolor='black', alpha=0.7, color='blue')
plt.axvline(media_gols, color='red', linestyle='dashed', linewidth=2, label=f'Média: {media_gols:.2f}')
plt.title("Distribuição do Número de Gols por Partida", fontsize=14)
plt.xlabel("Número de Gols", fontsize=12)
plt.ylabel("Frequência", fontsize=12)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Calcular a média de gols por ano
media_gols_por_ano = euro_data.groupby('year')['total_goals'].mean()

# Plotar a tendência da média de gols ao longo do tempo
plt.figure(figsize=(12, 6))
plt.plot(media_gols_por_ano.index, media_gols_por_ano.values, marker='o', linestyle='-', color='blue')
plt.title("Evolução da Média de Gols por Partida ao Longo do Tempo", fontsize=14)
plt.xlabel("Ano", fontsize=12)
plt.ylabel("Média de Gols por Partida", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(media_gols_por_ano.index, rotation=45)
plt.show()

print("Média de gols por ano:")
print(media_gols_por_ano)

## Comentário gráficos temporais

## Quais conclusões podemos tirar desses valores?

### Por exemplo, que o futebol tem se estabilizado com o tempo, mais iguadade na qualidade dos times. Bias

In [ ]:
# Determinar os vencedores finais da competição (um por ano)
vencedores_competicao = euro_data.groupby('year')['winner'].first()
vencedores_competicao

In [ ]:
# Contar o número de títulos por time
titulos_por_time = vencedores_competicao.value_counts()
titulos_por_time

In [ ]:
# Criar o gráfico de barras para visualizar os times com mais títulos
plt.figure(figsize=(12, 6))
titulos_por_time.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("Número de Títulos por Time na EURO", fontsize=14)
plt.xlabel("Times", fontsize=12)
plt.ylabel("Número de Títulos", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Gráfico ajustado com os anos de título na horizontal, empilhados com espaço entre eles
plt.figure(figsize=(12, 6))

# Criar o gráfico de barras
bars = plt.bar(titulos_por_time.index, titulos_por_time.values, color='skyblue', edgecolor='black')

# Adicionar os anos de vitória em cada barra, empilhados com espaço
for bar, country in zip(bars, titulos_por_time.index):
    anos_vitoria = vencedores_competicao[vencedores_competicao == country].index.tolist()
    for i, ano in enumerate(anos_vitoria):
        plt.text(
            bar.get_x() + bar.get_width() / 2,  # Posição central da barra
            i * 0.25 + 0.1,  # Espaço vertical entre os anos
            str(ano),
            ha='center', va='bottom', fontsize=10, rotation=0  # Texto na horizontal
        )

# Ajustar o gráfico
plt.title("Número de Títulos por País na EURO", fontsize=14)
plt.xlabel("País", fontsize=12)
plt.ylabel("Número de Títulos", fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.tight_layout()

# Exibir o gráfico
plt.show()




In [ ]:
import matplotlib.pyplot as plt

# Verificar se a coluna de público está presente no DataFrame
if 'match_attendance' in euro_data.columns:
    # Distribuição do público (histograma)
    plt.figure(figsize=(12, 6))
    plt.hist(euro_data['match_attendance'].dropna(), bins=20, color='skyblue', edgecolor='black', alpha=0.7)
    plt.title("Distribuição do Público nas Partidas da EURO", fontsize=14)
    plt.xlabel("Público", fontsize=12)
    plt.ylabel("Frequência", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # Média de público por ano
    publico_por_ano = euro_data.groupby('year')['match_attendance'].mean()
    plt.figure(figsize=(12, 6))
    plt.plot(publico_por_ano.index, publico_por_ano.values, marker='o', linestyle='-', color='blue')
    plt.title("Média de Público por Ano na EURO", fontsize=14)
    plt.xlabel("Ano", fontsize=12)
    plt.ylabel("Média de Público", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(publico_por_ano.index, rotation=45)
    plt.tight_layout()
    plt.show()

    # Partidas com maior público
    maior_publico = euro_data.nlargest(10, 'match_attendance')
    plt.figure(figsize=(12, 6))
    plt.barh(
        [f"{row['home_team']} vs {row['away_team']} ({row['year']})" for _, row in maior_publico.iterrows()],
        maior_publico['match_attendance'],
        color='skyblue',
        edgecolor='black'
    )
    plt.title("Top 10 Partidas com Maior Público na EURO", fontsize=14)
    plt.xlabel("Público", fontsize=12)
    plt.ylabel("Partidas", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("A coluna 'match_attendance' não está presente no DataFrame.")


In [ ]:
# Verificar se a coluna de público está presente no DataFrame
if 'match_attendance' in euro_data.columns:
    # Média de público por ano
    publico_por_ano = euro_data.groupby('year')['match_attendance'].mean()

    # País sede por ano (assumindo que existe uma coluna chamada 'host_country')
    pais_sede_por_ano = euro_data.groupby('year')['stadium_country_code'].first()

    # Criar o gráfico de média de público com o país sede
    plt.figure(figsize=(12, 6))
    plt.plot(publico_por_ano.index, publico_por_ano.values, marker='o', linestyle='-', color='blue', label="Média de Público")
    
    # Adicionar o país sede como rótulo
    for year, attendance in publico_por_ano.items():
        host_country = pais_sede_por_ano.get(year, "Desconhecido")
        plt.text(year, attendance + 1000, host_country, fontsize=9, ha='center', va='bottom', color='darkred')

    # Ajustar o gráfico
    plt.title("Média de Público por Ano na EURO (com País Sede)", fontsize=14)
    plt.xlabel("Ano", fontsize=12)
    plt.ylabel("Média de Público", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(publico_por_ano.index, rotation=45)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("A coluna 'match_attendance' ou 'stadium_country_code' não está presente no DataFrame.")


#### sistema de turismo da alemanha

### 1980 escandalo sobre partidas fixas na italia: https://en.wikipedia.org/wiki/1980_Totonero
### 1976: Yuguslavia pais socialista, pouco acesso 
### 1972 Recessão de 1973-1975: https://www.statista.com/topics/8095/the-1973-1975-recession/

In [ ]:
# 1980 escandalo sobre partidas fixas na italia: https://en.wikipedia.org/wiki/1980_Totonero
# 1976: Yuguslavia pais socialista, pouco acesso 
# 1972 Recessão de 1973-1975: https://www.statista.com/topics/8095/the-1973-1975-recession/

### Dados contam histórias. É nosso trabalho como cientistas de dados encontrar e contar essas histórias

# Análise cartões vermelhos

In [ ]:
# Na coluna de red_cards, há dicionários! Uma estrutura mais complexa para estrair os dados. Mas vamos lá!
valores_unicos = euro_data['red_cards'].unique()
print("Valores únicos na coluna 'red_cards':")
print(valores_unicos)

In [ ]:
import ast
from collections import Counter

In [ ]:
# Função para processar a coluna 'red_cards' e extrair informações detalhadas
def processar_cartoes_vermelhos(red_cards_column):
    contador_paises = Counter()
    contador_posicoes = Counter()
    contador_jogadores = Counter()
    
    for entry in red_cards_column:
        if pd.isna(entry):
            continue  # Ignorar NaNs
        
        try:
            # Converter a string em uma lista de dicionários
            cartoes = ast.literal_eval(entry)
            for cartao in cartoes:
                # Contar por país
                country_code = cartao.get('country_code', 'país não informado')
                contador_paises[country_code] += 1
                
                # Contar por posição
                position = cartao.get('national_field_position', 'posição não informada')
                contador_posicoes[position] += 1
                
        except (ValueError, SyntaxError):
            continue  # Ignorar entradas inválidas
    
    return contador_paises, contador_posicoes, contador_jogadores


In [ ]:
euro_data['red_cards'] = euro_data['red_cards'].fillna("[]")  # Substituir NaN por uma lista vazia
cartoes_por_pais, cartoes_por_posicao, cartoes_por_jogador = processar_cartoes_vermelhos(euro_data['red_cards'])

In [ ]:
cartoes_por_pais

In [ ]:
euro_data['red_cards'] = euro_data['red_cards'].fillna("[]")  # Substituir NaN por uma lista vazia
cartoes_por_pais, cartoes_por_posicao, cartoes_por_jogador = processar_cartoes_vermelhos(euro_data['red_cards'])

# Transformar os resultados em DataFrames
df_paises = pd.DataFrame.from_dict(cartoes_por_pais, orient='index', columns=['total_red_cards']).sort_values(by='total_red_cards', ascending=False)
df_posicoes = pd.DataFrame.from_dict(cartoes_por_posicao, orient='index', columns=['total_red_cards']).sort_values(by='total_red_cards', ascending=False)


# Criar gráficos diferentes
# Gráfico 1: Cartões por país
plt.figure(figsize=(12, 6))
df_paises.head(10).plot(kind='bar', color='red', edgecolor='black', legend=False)
plt.title("Top 10 Países com Mais Cartões Vermelhos na EURO", fontsize=14)
plt.xlabel("País", fontsize=12)
plt.ylabel("Número de Cartões Vermelhos", fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.tight_layout()
plt.show()

# Gráfico 2: Cartões por posição
plt.figure(figsize=(12, 6))
df_posicoes.plot(kind='bar', color='orange', edgecolor='black', legend=False)
plt.title("Cartões Vermelhos por Posição dos Jogadores na EURO", fontsize=14)
plt.xlabel("Posição", fontsize=12)
plt.ylabel("Número de Cartões Vermelhos", fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.tight_layout()
plt.show()


  ### 1. Por que parecia ter menos cartões por país?                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                              
  • No gráfico de países: O seu código usa  df_paises.head(10).plot(...) . Isso filtra apenas os 10 países com mais cartões vermelhos. A soma desses top 10 países dá exatamente 28 cartões.                                                                                                                  
  • No gráfico de posições: O código plota o DataFrame  df_posicoes  inteiro (sem  .head(10) ). Como existem apenas 5 categorias de posições no dataset, o gráfico mostra todas elas, somando os 48 cartões totais.                                                                                           
  • Se você somar a coluna  total_red_cards  inteira de ambos os DataFrames (sem limitar com  .head ), verá que ambas contam exatamente os mesmos 48 cartões vermelhos.    

## o que o antigravity (gemini) corrigiu do codigo anterior:
### 2. Oportunidades de melhorias identificadas no seu código:                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                                                                              
  1. Contagem de jogadores ( contador_jogadores ): Na sua função original, você define e retorna  contador_jogadores , mas esqueceu de incrementá-lo dentro do loop de cartões.                                                                                                                               
  2. Posições vazias ( '' ): O campo  national_field_position  vem vazio ( '' ) para 7 jogadores na base de dados, gerando uma barra sem rótulo no seu gráfico de posições. É melhor renomear esses casos para  'posição não informada' .     

In [ ]:
def processar_cartoes_vermelhos(red_cards_column):                                                                                                                                                                                                                                                        
    contador_paises = Counter()                                                                                                                                                                                                                                                                           
    contador_posicoes = Counter()                                                                                                                                                                                                                                                                         
    contador_jogadores = Counter()                                                                                                                                                                                                                                                                        
                                                                                                                                                                                                                                                                                                            
    for entry in red_cards_column:                                                                                                                                                                                                                                                                        
        if pd.isna(entry):                                                                                                                                                                                                                                                                                
            continue  # Ignorar NaNs                                                                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                                                                            
        try:                                                                                                                                                                                                                                                                                              
            # Converter a string em uma lista de dicionários                                                                                                                                                                                                                                              
            cartoes = ast.literal_eval(entry)                                                                                                                                                                                                                                                             
            for cartao in cartoes:                                                                                                                                                                                                                                                                        
                # Contar por país                                                                                                                                                                                                                                                                         
                country_code = cartao.get('country_code', 'país não informado')                                                                                                                                                                                                                           
                contador_paises[country_code] += 1                                                                                                                                                                                                                                                        
                                                                                                                                                                                                                                                                                                            
                # Contar por posição (tratar string vazia como não informada)                                                                                                                                                                                                                             
                position = cartao.get('national_field_position', '')                                                                                                                                                                                                                                      
                if position == '':                                                                                                                                                                                                                                                                        
                    position = 'posição não informada'                                                                                                                                                                                                                                                    
                contador_posicoes[position] += 1                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                            
                # Contar por jogador (que estava faltando no seu código anterior)                                                                                                                                                                                                                         
                jogador = cartao.get('international_name', 'jogador não informado')                                                                                                                                                                                                                       
                contador_jogadores[jogador] += 1                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                            
        except (ValueError, SyntaxError):                                                                                                                                                                                                                                                                 
            continue  # Ignorar entradas inválidas                                                                                                                                                                                                                                                        
                                                                                                                                                                                                                                                                                                            
    return contador_paises, contador_posicoes, contador_jogadores                                                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                                            
#### Substitua a célula de Plotagem de Gráficos por:                                                                                                                                                                                                                                                        
                                                                                                                                                                                                                                                                                                            
euro_data['red_cards'] = euro_data['red_cards'].fillna("[]")  # Substituir NaN por uma lista vazia                                                                                                                                                                                                        
cartoes_por_pais, cartoes_por_posicao, cartoes_por_jogador = processar_cartoes_vermelhos(euro_data['red_cards'])                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                            
# Transformar os resultados em DataFrames                                                                                                                                                                                                                                                                 
df_paises = pd.DataFrame.from_dict(cartoes_por_pais, orient='index', columns=['total_red_cards']).sort_values(by='total_red_cards', ascending=False)                                                                                                                                                      
df_posicoes = pd.DataFrame.from_dict(cartoes_por_posicao, orient='index', columns=['total_red_cards']).sort_values(by='total_red_cards', ascending=False)                                                                                                                                                 
                                                                                                                                                                                                                                                                                                            
# Linhas de debug para você conferir que as contagens agora batem 100%                                                                                                                                                                                                                                    
print(f"Total de cartões por país (base inteira): {df_paises['total_red_cards'].sum()}")                                                                                                                                                                                                                  
print(f"Total de cartões por posição (base inteira): {df_posicoes['total_red_cards'].sum()}")                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                                                            
# Gráfico 1: Cartões por país (exibindo todos os países para ver a soma de 48)
plt.figure(figsize=(12, 6))
df_paises.plot(kind='bar', color='red', edgecolor='black', legend=False) 
plt.title("Cartões Vermelhos por País na EURO", fontsize=14)
plt.xlabel("País", fontsize=12)
plt.ylabel("Número de Cartões Vermelhos", fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.tight_layout()
plt.show()

# Gráfico 2: Cartões por posição
plt.figure(figsize=(12, 6))
df_posicoes.plot(kind='bar', color='orange', edgecolor='black', legend=False)
plt.title("Cartões Vermelhos por Posição dos Jogadores na EURO", fontsize=14)
plt.xlabel("Posição", fontsize=12)
plt.ylabel("Número de Cartões Vermelhos", fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.tight_layout()
plt.show()

# Vices

In [ ]:
# # Filtrar as partidas finais
# finais = euro_data[euro_data['round'] == 'FINAL']

# Para evitar o copy warning:
# cópia explícita
finais = euro_data[euro_data['round'] == 'FINAL'].copy()

# Identificar os vice-campeões (times que jogaram na final, mas não venceram)
finais.loc[:, 'loser'] = finais.apply(
    lambda row: row['home_team'] if row['away_team'] == row['winner'] else row['away_team'],
    axis=1
)


# Contar os vice-campeões
vice_campeoes = finais['loser'].value_counts()

# Criar um dicionário para armazenar os anos de vice-campeonatos por time
anos_vices = finais.groupby('loser')['year'].apply(list)

# Visualizar os resultados com os anos no gráfico
plt.figure(figsize=(12, 6))
bars = plt.bar(vice_campeoes.index, vice_campeoes.values, color='gold', edgecolor='black')

# Ajustar o limite superior do gráfico para acomodar os anos
plt.ylim(0, vice_campeoes.max() + 2)

# Adicionar os anos de vice-campeonato como rótulos nas barras
for bar, team in zip(bars, vice_campeoes.index):
    anos_texto = ', '.join(map(str, anos_vices[team]))
    plt.text(
        bar.get_x() + bar.get_width() / 2,  # Centralizar o texto na barra
        bar.get_height() + 0.2,  # Posicionar logo acima da barra
        anos_texto,
        ha='center', va='bottom', fontsize=10, rotation=45  # Rotacionar texto em 45 graus
    )

# Ajustar o gráfico
plt.title("Times Mais Vice-Campeões da EURO (com Anos)", fontsize=14)
plt.xlabel("Time", fontsize=12)
plt.ylabel("Número de Vices", fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Análise de Desempenho em Finais
# Filtrar as partidas finais
finais = euro_data[euro_data['round'] == 'FINAL']

# Contar número de finais jogadas por time
finais_jogadas = finais['home_team'].value_counts() + finais['away_team'].value_counts()

# Contar vitórias em finais
vitorias_finais = finais['winner'].value_counts()

# Combinar as duas contagens para criar uma análise completa
desempenho_finais = pd.DataFrame({
    'finais_jogadas': finais_jogadas,
    'vitorias_finais': vitorias_finais
}).fillna(0)

# Adicionar uma coluna de derrotas em finais
desempenho_finais['derrotas_finais'] = desempenho_finais['finais_jogadas'] - desempenho_finais['vitorias_finais']

# Exibir a tabela de desempenho em finais
print("Desempenho em Finais:")
print(desempenho_finais.sort_values(by='vitorias_finais', ascending=False))

# Gráfico de desempenho em finais
desempenho_finais.sort_values(by='finais_jogadas', ascending=False).plot(
    kind='bar',
    figsize=(12, 6),
    stacked=True,
    color=['blue', 'gold', 'red'],
    edgecolor='black'
)
plt.title("Desempenho em Finais da EURO", fontsize=14)
plt.xlabel("Seleção", fontsize=12)
plt.ylabel("Número de Finais", fontsize=12)
plt.legend(['Finais Jogadas', 'Vitórias', 'Derrotas'], loc='upper right')
plt.xticks(rotation=45, fontsize=10)
plt.tight_layout()
plt.show()
